In [4]:
# =========================
# Normal LwF (CIFAR-100 50/50 split) - Kaggle Ready
# =========================

import os
import copy
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
SEED = 42
BATCH_SIZE = 128
NUM_WORKERS = 2
EPOCHS_A = 70           # task A training
EPOCHS_B = 90           # task B + LwF
LR_A = 0.1
LR_B = 0.05
WEIGHT_DECAY = 5e-4

# LwF params
LWF_LAMBDA = 1.0        # distillation weight
TEMPERATURE = 2.0       # distillation temperature

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -------------------------
# Reproducibility
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# -------------------------
# Data
# -------------------------
mean = (0.5071, 0.4867, 0.4408)
std  = (0.2675, 0.2565, 0.2761)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

root = "/kaggle/input/cifar-100" if os.path.exists("/kaggle/input/cifar-100") else "./data"

train_ds = torchvision.datasets.CIFAR100(root=root, train=True, download=True, transform=train_tf)
test_ds  = torchvision.datasets.CIFAR100(root=root, train=False, download=True, transform=test_tf)

def class_subset(dataset, class_range):
    idx = [i for i, y in enumerate(dataset.targets) if y in class_range]
    return Subset(dataset, idx)

train_A = class_subset(train_ds, range(0, 50))
train_B = class_subset(train_ds, range(50, 100))
test_A  = class_subset(test_ds, range(0, 50))
test_B  = class_subset(test_ds, range(50, 100))

train_loader_A = DataLoader(train_A, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
train_loader_B = DataLoader(train_B, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
test_loader_A  = DataLoader(test_A, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader_B  = DataLoader(test_B, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# -------------------------
# Model (normal ResNet18 for CIFAR)
# -------------------------
class ResNet18CIFAR(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        m = torchvision.models.resnet18(weights=None)
        m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        m.maxpool = nn.Identity()
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        self.net = m

    def forward(self, x):
        return self.net(x)

model = ResNet18CIFAR(num_classes=100).to(DEVICE)

# -------------------------
# Eval helpers
# -------------------------
@torch.no_grad()
def eval_task_aware(model, loader, task):
    # task='A' -> only classes 0..49
    # task='B' -> only classes 50..99
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        if task == "A":
            pred = logits[:, :50].argmax(1)                   # 0..49
        else:
            pred = logits[:, 50:].argmax(1) + 50              # 50..99
        correct += (pred == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total

@torch.no_grad()
def eval_cil(model, loader):
    # true class-incremental (argmax over all 100 classes)
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total

# -------------------------
# Phase 1: Train Task A
# -------------------------
optimizer = torch.optim.SGD(model.parameters(), lr=LR_A, momentum=0.9, weight_decay=WEIGHT_DECAY, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_A)

for ep in range(EPOCHS_A):
    model.train()
    for x, y in train_loader_A:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)  # normal CE
        loss.backward()
        optimizer.step()

    scheduler.step()

    if (ep + 1) % 10 == 0:
        a_ta = eval_task_aware(model, test_loader_A, "A")
        print(f"[Phase1][{ep+1:03d}/{EPOCHS_A}] A(TA): {a_ta:.2f}")

acc_A_init_ta = eval_task_aware(model, test_loader_A, "A")
print(f"\nTask A init (TA): {acc_A_init_ta:.2f}")

# Freeze teacher after phase 1
teacher = copy.deepcopy(model).to(DEVICE)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

# -------------------------
# Phase 2: Train Task B with LwF
# -------------------------
optimizer = torch.optim.SGD(model.parameters(), lr=LR_B, momentum=0.9, weight_decay=WEIGHT_DECAY, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_B)

T = TEMPERATURE
lam = LWF_LAMBDA

for ep in range(EPOCHS_B):
    model.train()
    for x, y in train_loader_B:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        student_logits = model(x)
        with torch.no_grad():
            teacher_logits = teacher(x)

        # CE on new classes (real labels 50..99)
        ce_new = F.cross_entropy(student_logits, y)

        # LwF distillation on old classes only (0..49)
        s_old = student_logits[:, :50] / T
        t_old = teacher_logits[:, :50] / T

        kd = F.kl_div(
            F.log_softmax(s_old, dim=1),
            F.softmax(t_old, dim=1),
            reduction="batchmean"
        ) * (T * T)

        loss = ce_new + lam * kd
        loss.backward()
        optimizer.step()

    scheduler.step()

    if (ep + 1) % 10 == 0:
        a_cil = eval_cil(model, test_loader_A)
        b_cil = eval_cil(model, test_loader_B)
        print(f"[Phase2][{ep+1:03d}/{EPOCHS_B}] A(CIL): {a_cil:.2f} | B(CIL): {b_cil:.2f}")

# -------------------------
# Final Report
# -------------------------
acc_A_final_ta = eval_task_aware(model, test_loader_A, "A")
acc_B_final_ta = eval_task_aware(model, test_loader_B, "B")
acc_A_final_cil = eval_cil(model, test_loader_A)
acc_B_final_cil = eval_cil(model, test_loader_B)

ret_ta = (acc_A_final_ta / acc_A_init_ta) * 100 if acc_A_init_ta > 0 else 0.0
ret_cil = (acc_A_final_cil / acc_A_init_ta) * 100 if acc_A_init_ta > 0 else 0.0

print("\n" + "=" * 80)
print("NORMAL LwF RESULTS (Single Seed)")
print("=" * 80)
print(f"A Init (Task-aware) : {acc_A_init_ta:.2f}")
print(f"A Final (Task-aware): {acc_A_final_ta:.2f}")
print(f"B Final (Task-aware): {acc_B_final_ta:.2f}")
print(f"A Final (CIL)       : {acc_A_final_cil:.2f}")
print(f"B Final (CIL)       : {acc_B_final_cil:.2f}")
print(f"Retention (TA)      : {ret_ta:.2f}%")
print(f"Retention (CIL)     : {ret_cil:.2f}%")
print("=" * 80)

Device: cuda
[Phase1][010/70] A(TA): 58.22
[Phase1][020/70] A(TA): 65.32
[Phase1][030/70] A(TA): 68.56
[Phase1][040/70] A(TA): 71.78
[Phase1][050/70] A(TA): 79.16
[Phase1][060/70] A(TA): 80.16
[Phase1][070/70] A(TA): 80.72

Task A init (TA): 80.72
[Phase2][010/90] A(CIL): 0.52 | B(CIL): 69.36
[Phase2][020/90] A(CIL): 0.90 | B(CIL): 70.76
[Phase2][030/90] A(CIL): 2.22 | B(CIL): 73.40
[Phase2][040/90] A(CIL): 3.18 | B(CIL): 74.98
[Phase2][050/90] A(CIL): 5.50 | B(CIL): 77.42
[Phase2][060/90] A(CIL): 7.70 | B(CIL): 77.94
[Phase2][070/90] A(CIL): 10.02 | B(CIL): 78.50
[Phase2][080/90] A(CIL): 11.64 | B(CIL): 78.88
[Phase2][090/90] A(CIL): 11.46 | B(CIL): 78.78

NORMAL LwF RESULTS (Single Seed)
A Init (Task-aware) : 80.72
A Final (Task-aware): 78.34
B Final (Task-aware): 78.80
A Final (CIL)       : 11.46
B Final (CIL)       : 78.78
Retention (TA)      : 97.05%
Retention (CIL)     : 14.20%
